<a href="https://colab.research.google.com/github/matheusvendas/Marketing-Mix-Modeling/blob/main/Marketing_Mix_Modeling.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MMM Tradicional (cenário perfeito)

In [45]:
import pandas as pd
import numpy as np
import statsmodels.api as sm

# ---------------------------------------------------------
# 1. GERANDO DADOS FICTÍCIOS (104 semanas = 2 anos)
# ---------------------------------------------------------
np.random.seed(42) # Para garantir que o resultado seja o mesmo sempre
semanas = 104

# Investimentos em Mídia (Features) - Valores em Reais (R$)
tv_spend = np.random.uniform(10000, 50000, semanas)
google_spend = np.random.uniform(5000, 30000, semanas)
meta_spend = np.random.uniform(2000, 15000, semanas)

# Variável Exógena (Sazonalidade - ex: vendas de antigripais no inverno)
# Usando uma onda senoidal para simular os altos e baixos do ano
sazonalidade = np.sin(np.linspace(0, 4 * np.pi, semanas)) + 1

In [46]:
# ---------------------------------------------------------
# 2. SIMULANDO A REALIDADE (A "Verdade" que o modelo tentará descobrir)
# ---------------------------------------------------------
# Vamos definir o ROI "verdadeiro" escondido nos dados:
baseline = 50000       # Venda base (inércia da farmácia sem anúncios)
roi_tv = 1.2           # Cada R$1 na TV traz R$ 1.20
roi_google = 2.5       # Cada R$1 no Google traz R$ 2.50
roi_meta = 4.0       # Cada R$1 no Retail Media traz R$ 4.00
fator_sazonal = 8000   # Peso do inverno nas vendas

ruido = np.random.normal(0, 4000, semanas)
vendas = (baseline +
          (tv_spend * roi_tv) +
          (google_spend * roi_google) +
          (meta_spend * roi_meta) +
          (sazonalidade * fator_sazonal) +
          ruido)

df = pd.DataFrame({
    'Vendas': vendas,
    'TV_Spend': tv_spend,
    'Google_Spend': google_spend,
    'Meta_spend': meta_spend,
    'Sazonalidade': sazonalidade
})

In [47]:
# ---------------------------------------------------------
# 3. RODANDO O MMM TRADICIONAL (Regressão Linear Múltipla OLS)
# ---------------------------------------------------------
# Separando o que é causa (X) e o que é efeito (y)
X = df[['TV_Spend', 'Google_Spend', 'Meta_spend', 'Sazonalidade']]
X = sm.add_constant(X) # Obrigatório: Adiciona o intercepto (o modelo precisa estimar o Baseline)
y = df['Vendas']

# Treinando o modelo
modelo = sm.OLS(y, X).fit()

print(modelo.summary())

                            OLS Regression Results                            
Dep. Variable:                 Vendas   R-squared:                       0.983
Model:                            OLS   Adj. R-squared:                  0.983
Method:                 Least Squares   F-statistic:                     1458.
Date:                Sat, 23 May 2026   Prob (F-statistic):           5.09e-87
Time:                        18:53:59   Log-Likelihood:                -1001.0
No. Observations:                 104   AIC:                             2012.
Df Residuals:                      99   BIC:                             2025.
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                   coef    std err          t      P>|t|      [0.025      0.975]
--------------------------------------------------------------------------------
const         5.011e+04   1652.296     30.328   

# MMM Tradicional com complexidade de correlação entre erro e variáveis


In [48]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
import pymc as pm

# =========================================================
# 1. GERANDO DADOS COM O VIÉS DE SELEÇÃO
# =========================================================
np.random.seed(42)
semanas = 104

# A Intenção de Compra é o Epsilon (ruído oculto).
# O modelo nunca vai ver essa variável, ela só existe na realidade.
intencao_oculta = np.random.uniform(0, 10, semanas)

# Investimentos Base
tv_spend = np.random.uniform(10000, 50000, semanas)
meta_spend = np.random.uniform(2000, 15000, semanas)
sazonalidade = np.sin(np.linspace(0, 4 * np.pi, semanas)) + 1

# O PROBLEMA: O algoritmo do Google Ads gasta mais dinheiro exatamente
# quando a intenção de compra está alta. (Correlação entre X e Ruído)
google_spend = np.random.uniform(5000, 15000, semanas) + (intencao_oculta * 1500)

# (gabarito)
baseline = 50000
roi_tv = 1.2
roi_google = 2.5       # Este é o ROI real e incremental do Google
roi_retail = 4.0
fator_sazonal = 8000
fator_intencao = 3000

# O faturamento real do caixa da farmácia
vendas = (baseline +
          (tv_spend * roi_tv) +
          (google_spend * roi_google) +
          (meta_spend * roi_retail) +
          (sazonalidade * fator_sazonal) +
          (intencao_oculta * fator_intencao) + # Isso fica oculto
          np.random.normal(0, 2000, semanas))

# O Varejista só tem estas colunas. A 'intencao_oculta' não está na planilha.
df = pd.DataFrame({
    'Vendas': vendas,
    'TV_Spend': tv_spend,
    'Google_Spend': google_spend,
    'Meta_Spend': meta_spend,
    'Sazonalidade': sazonalidade
})

# =========================================================
# 2. O PROBLEMA NO OLS TRADICIONAL
# =========================================================
X = df[['TV_Spend', 'Google_Spend', 'Meta_Spend', 'Sazonalidade']]
X = sm.add_constant(X)
modelo_ols = sm.OLS(df['Vendas'], X).fit()

print("--- RESULTADO OLS (O Painel Mente) ---")
roi_google_ilusorio = modelo_ols.params['Google_Spend']
print(f"ROI Google Calculado (OLS): {roi_google_ilusorio}")
# O OLS vai cuspir um ROI de ~4 (Superestimou, a verdade é 2.5)
print(f"Baseline Calculado (OLS): {modelo_ols.params['const']:.0f}")
# O OLS vai estimar o Baseline em ~36.000 (roubou vendas orgânicas, a verdade é 50.000)

--- RESULTADO OLS (O Painel Mente) ---
ROI Google Calculado (OLS): 3.9192641946358635
Baseline Calculado (OLS): 38704


# MMM Bayesiano corrigindo problemas com Priors (experimentos prévios)

In [49]:
import pymc as pm
import arviz as az

print("--- INICIANDO MOTOR BAYESIANO ---")

with pm.Model() as mmm_bayesiano:

    # ---------------------------------------------------------
    # 1. AS PRIORS (experimento prévio)
    # ---------------------------------------------------------
    # Para o Baseline, TV, meta e Sazonalidade, nós não temos experimentos.
    # Então damos distribuições largas (muita incerteza), deixando os dados falarem.
    beta_baseline = pm.Normal("Baseline", mu=40000, sigma=20000)
    beta_tv = pm.HalfNormal("ROI_TV", sigma=3)
    beta_meta = pm.HalfNormal("ROI_Meta", sigma=5)
    beta_sazonal = pm.Normal("Fator_Sazonal", mu=5000, sigma=5000)

    # --- REDUZINDO O VIÉS ---
    # Nós informamos o resultado do nosso teste de CDP.
    # mu=2.5 (o valor do teste) e sigma=0.05 (Altíssima certeza, não deixe o modelo fugir!)
    beta_google = pm.Normal("ROI_Google", mu=2.5, sigma=0.05)

    # O ruído do modelo
    sigma_erro = pm.HalfNormal("Ruido", sigma=5000)

    # ---------------------------------------------------------
    # 2. A EQUAÇÃO
    # ---------------------------------------------------------
    # Multiplicamos as nossas Priors probabilísticas pelos dados reais da planilha (.values)
    vendas_esperadas = (
        beta_baseline +
        (beta_tv * df['TV_Spend'].values) +
        (beta_google * df['Google_Spend'].values) +
        (beta_meta * df['Meta_Spend'].values) +
        (beta_sazonal * df['Sazonalidade'].values)
    )

    # ---------------------------------------------------------
    # 3. VEROSSIMILHANÇA
    # ---------------------------------------------------------
    obs = pm.Normal("Vendas_Obs",
                    mu=vendas_esperadas,
                    sigma=sigma_erro,
                    observed=df['Vendas'].values) # Travando na realidade

    # ---------------------------------------------------------
    # 4. AMOSTRAGEM (MCMC)
    # ---------------------------------------------------------
    # O algoritmo vai testar milhares de cenários
    trace = pm.sample(draws=1500, tune=1000, cores=1, target_accept=0.9, random_seed=42, progressbar=True)

print("\n--- RESULTADO FINAL CALIBRADO ---")
resumo = az.summary(trace, var_names=["Baseline", "ROI_Google", "ROI_TV", "ROI_Meta"])
roi_google_bayes = resumo['mean']['ROI_Google']
print(resumo[['mean', 'sd', 'hdi_3%', 'hdi_97%']])

--- INICIANDO MOTOR BAYESIANO ---


Output()


--- RESULTADO FINAL CALIBRADO ---
                 mean        sd     hdi_3%    hdi_97%
Baseline    61194.702  3438.023  55077.185  67975.077
ROI_Google      2.635     0.051      2.546      2.740
ROI_TV          1.245     0.072      1.102      1.368
ROI_Meta        4.039     0.243      3.588      4.498


# Conclusão

In [50]:
print("Roi real do Google:", roi_google)
print()
print("Roi do Google sem Bayes", roi_google_ilusorio)
print()
print("Roi do google controlado por Bayes", roi_google_bayes)

Roi real do Google: 2.5

Roi do Google sem Bayes 3.9192641946358635

Roi do google controlado por Bayes 2.635
